In [1]:
import os
import json
import hmac
import time
import base64
import hashlib
import requests
from dotenv import load_dotenv
from datetime import datetime, timezone
from urllib.parse import urlencode, quote, unquote
from email.utils import format_datetime

load_dotenv()
pudu_api_key = os.getenv('Pd_key')
pudu_api_secret = os.getenv('Pd_secret')
Aurotek_id = os.getenv('Aurotek_id')
BellabotPro = "8BR054C02050014"
Flashbot = "8FF055923050007"

In [2]:
class PuduAuth:

    @staticmethod
    def utc_date():
        return format_datetime(datetime.now(timezone.utc), usegmt=True)

    @staticmethod
    def md5_base64(content: str) -> str:
        md5_hex = hashlib.md5(content.encode("utf-8")).hexdigest()
        return base64.b64encode(md5_hex.encode("utf-8")).decode("utf-8")

    @staticmethod
    def generate_signature(
        secret: str,
        method: str,
        path: str,
        x_date: str,
        content_md5: str = ""
    ):
        string_to_sign = (
            f"x-date: {x_date}\n"
            f"{method.upper()}\n"
            f"application/json\n"
            f"application/json\n"
            f"{content_md5}\n"
            f"{path}"
        )
        signature = hmac.new(secret.encode("utf-8"), string_to_sign.encode("utf-8"), hashlib.sha1).digest()
        return base64.b64encode(signature).decode("utf-8")

class PuduApiClient:

    BASE_URL = ("https://css-open-platform.pudutech.com") # /pudu-entry" # "https://css-open-platform.pudutech.com"
    
    def __init__(
        self,
        app_key: str,
        app_secret: str,
        timeout=30
    ):
        self.app_key = app_key
        self.app_secret = app_secret
        self.timeout = timeout

    def _build_path(
        self,
        endpoint,
        params=None
    ):
        if not params:
            return endpoint
        sorted_params = sorted(params.items(), key=lambda x: x[0])               
        query = "&".join(f"{k}={v}" for k, v in sorted_params)        
        #query = unquote(urlencode(sorted_params, doseq=True))
        return f"{endpoint}?{query}"

    def _headers(
        self,
        method,
        path,
        body=""
    ):
        # print(path) # 為了看 open_map url path 的差別
        x_date = PuduAuth.utc_date()
        content_md5 = ""
        if method.upper() == "POST":
            if body:
                content_md5 = (PuduAuth.md5_base64(body))
        signature = (
            PuduAuth.generate_signature(
                self.app_secret,
                method,
                path,
                x_date,
                content_md5
            )
        )
        authorization = (
            f'hmac id="{self.app_key}", '
            f'algorithm="hmac-sha1", '
            f'headers="x-date", '
            f'signature="{signature}"'
        )
        headers = {
            "Accept": "application/json",
            "Content-Type": "application/json",
            "x-date": x_date,
            "Authorization": authorization
        }
        if content_md5:
            headers["Content-MD5"] = content_md5
        return headers

    def _get_v2(self, endpoint, params=None):
        # ✅ 簽名用「原始未編碼」的 query 字串（依 key 排序，與成功 API 一致）
        if params:
            sorted_items = sorted(params.items(), key=lambda x: x[0])
            raw_query = "&".join(f"{k}={v}" for k, v in sorted_items)
            sign_path = f"{endpoint}?{raw_query}"
        else:
            sign_path = endpoint

        headers = self._headers("GET", sign_path)
        print(headers)
        url = self.BASE_URL + endpoint

        try:
            r = requests.get(
                url,
                params=params,        # ✅ 送出時才由 requests 編碼
                headers=headers,
                timeout=self.timeout
            )
            print("\n===== DEBUG =====")
            print("SIGN PATH:", sign_path)
            print("REAL URL :", r.url)
            print("STATUS   :", r.status_code)
            print("TEXT     :", r.text)
            r.raise_for_status()
            return r.json()
        except Exception as err:
            print(f"\n❌ API 請求失敗，錯誤原因: {err}")
            return {"success": False, "error_msg": str(err), "data": []}

    def _get(
        self,
        endpoint,
        params=None
    ):
        path = self._build_path(endpoint, params)
        headers = self._headers("GET", path)        
        url = self.BASE_URL + endpoint

        try:
            r = requests.get(
                url,
                params=params,
                headers=headers,
                timeout=self.timeout
            )
            r.raise_for_status()
            return r.json()
        except Exception as err:
            print(f"\n❌ API 請求失敗，錯誤原因: {err}")
            return {
                "success": False, 
                "error_msg": str(err), 
                "data": []  # 👈 塞一個空列表防呆，確保後面調用 .get('data') 不會崩潰
            }

    def _post(self, endpoint, payload):
        body = json.dumps(
            payload,
            ensure_ascii=False,
            separators=(",", ":")
        )

        headers = self._headers(
            "POST",
            endpoint,
            body
        )

        url = self.BASE_URL + endpoint

        try:
            r = requests.post(
                url,
                data=body.encode("utf-8"),
                headers=headers,
                timeout=self.timeout
            )

            # 🔥 先印出 response（超重要）
            print("\n=== POST DEBUG ===")
            print("URL:", url)
            print("BODY:", body)
            print("HEADERS:", headers)
            print("STATUS:", r.status_code)
            print("RESPONSE:", r.text)

            r.raise_for_status()

            return r.json()

        except Exception as err:
            print(f"\n❌ POST API 請求失敗: {err}")

            return {
                "success": False,
                "error_msg": str(err),
                "status_code": getattr(r, "status_code", None) if 'r' in locals() else None,
                "response": r.text if 'r' in locals() else None,  # 🔥 超關鍵
                "data": []
            }


    def _post_v1(
        self,
        endpoint,
        payload
    ):
        body = json.dumps(
            payload,
            ensure_ascii=False,
            separators=(",", ":")
        )
        headers = self._headers(
            "POST",
            endpoint,
            body
        )
        url = self.BASE_URL + endpoint
        r = requests.post(
            url,
            data=body.encode("utf-8"),
            headers=headers,
            timeout=self.timeout
        )
        r.raise_for_status()
        return r.json()

    def get_shops(
        self,
        limit=100,
        offset=0
    ):
        return self._get(
            "/pudu-entry/data-open-platform-service/v1/api/shop",
            {
                "limit": limit,
                "offset": offset
            }
        )

    def get_points(
        self,
        sn,
        limit=100,
        offset=0,
    ):
        return self._get(
            "/pudu-entry/map-service/v1/open/point",
            {
                "sn": sn,
                "limit": limit,
                "offset": offset
            }
        )

    def get_robots(
        self,
        shop_id
    ):
        return self._get(
            "/pudu-entry/data-open-platform-service/v1/api/robot",
            {
                "shop_id": shop_id
            }
        )

    def get_maps(
        self,
        shop_id
    ):
        return self._get(
            "/pudu-entry/data-open-platform-service/v1/api/maps",
            {
                "shop_id": shop_id
            }
        )

    def get_map_detail(
        self,
        shop_id,
        map_name,
        width=1200,
        height=800
    ):
        return self._get(
            "/pudu-entry/data-open-platform-service/v1/api/map",
            {
                "shop_id": shop_id,
                "map_name": map_name,
                "device_width": width,
                "device_height": height
            }
        )

    def get_robot_position(
        self,
        shop_id,
        sn
    ):
        return self._get(
            "/pudu-entry/data-open-platform-service/v1/api/map/robotCurrentPosition",
            {
                "shop_id": shop_id,
                "sn": sn
            }
        )

    def get_position(
        self,        
        sn
    ):
        return self._get(
            "/pudu-entry/open-platform-service/v1/robot/get_position",
            {                
                "sn": sn,
            }
        )

    def open_map(
        self,
        shop_id,
        map_name        
    ):
        return self._get(
            "/pudu-entry/map-service/v1/open/map",
            {
                "shop_id": shop_id,                
                "map_name": map_name
            }
        )

    def get_voice_list(
        self,        
        sn
    ):
        return self._get(
            "/pudu-entry/open-platform-service/v1/voice/list",
            {                
                "sn": sn,
            }
        )

    def get_map_list(self, sn):
        return self._get(
            "/pudu-entry/map-service/v1/open/list",
            {
                "sn": sn
            }
        )

    def get_map_current(self, sn, need_element=False):
        return self._get(
            "/pudu-entry/map-service/v1/open/current",
            {
                "sn": sn,
                "need_element": need_element
            }
        )   
        
    def switch_map(
        self,
        sn,
        map_name,
        point,
        call_device_name="PythonSDK"
    ):
        return self._post(
            "/pudu-entry/open-platform-service/v1/switch_map",
            {
                "sn": sn,
                "map_name": map_name,
                "point": point,
                "call_device_name": call_device_name
            }
        )

    def switch_map_in_elevator(
        self,
        sn,
        map_name,
        point,
        call_device_name="PythonSDK"
    ):
        return self._post(
            "/pudu-entry/open-platform-service/v1/robot/map/switch_in_elevator",
            {
                "sn": sn,
                "map_name": map_name,
                "point": point,
                "call_device_name": call_device_name
            }
        )

    def create_transport_task(self, payload):
        return self._post(
            "/pudu-entry/open-platform-service/v1/transport_task",
            payload
        )

    def create_delivery_task(self, payload):
        return self._post(
            "/pudu-entry/open-platform-service/v1/delivery_task",
            payload
        )

    def get_robot_current_position(self, shop_id, sn):
        return self._get(
            "/pudu-entry/data-open-platform-service/v1/api/map/robotCurrentPosition",
            {
                "shop_id": shop_id,
                "sn": sn
            }
        )
    
    def get_door_state(
        self,
        sn,
        ):
        return self._get(
            "/pudu-entry/open-platform-service/v1/door_state",
            {
                "sn": sn,
            }
        )

    def get_by_sn(self, sn):
        return self._get(
            "/pudu-entry/open-platform-service/v1/status/get_by_sn",
            {
                "sn": sn,
            } 
        )

    def get_by_sn2(self, sn):
        return self._get(
            "/pudu-entry/open-platform-service/v2/status/get_by_sn",
            {
                "sn": sn,
            }             
        )

    def state_get(self, sn):
        return self._get(
            "/pudu-entry/open-platform-service/v1/robot/task/state/get",
            {
                "sn": sn,
            }            
        )

    def call_list(self, sn):
        return self._get(
            "/pudu-entry/open-platform-service/v1/call/list",
            {
                "sn": sn,
            }             
        )

    def recharge(self, sn):
        return self._get(
            "/pudu-entry/open-platform-service/v2/recharge",
            {
                "sn": sn,
            }            
        )

    def cancel_task(self, payload):
        return self._post(
            "/pudu-entry/open-platform-service/v1/cancel_task",
            payload
        )

    def screen_set(self, payload):
        return self._post(
            "/pudu-entry/open-platform-service/v1/robot/screen/set",
            payload
        )

    def custom_call_cancel(self, payload):
        return self._post(
            "/pudu-entry/open-platform-service/v1/custom_call/cancel",
            payload
        )

    def control_doors(self, payload):
        return self._post(
            "/pudu-entry/open-platform-service/v1/control_doors",
            payload
        )

    def custom_content(self, payload):
        return self._post(
            "/pudu-entry/open-platform-service/v1/custom_content",
            payload
        )

    def custom_complete(self, payload):
        return self._post(
            "/pudu-entry/open-platform-service/v1/custom_call/complete",
            payload
        )    

    def custom_call2(self, payload):
        return self._post(
            "/pudu-entry/open-platform-service/v1/custom_call",
            payload
        )

    def custom_call(
        self,
        sn,
        shop_id,
        map_name,
        point,
        point_type="table",
        call_device_name="PythonSDK",
        call_mode="IMG",
        mode_data=None,
        do_not_queue=False,
        robot_group_ids=None,
        filter_category_ids=None,
        priority=1
    ):
        payload = {
            "sn": sn,
            "shop_id": shop_id,
            "map_name": map_name,
            "point": point,
            "point_type": point_type,
            "call_device_name": call_device_name,
            "call_mode": call_mode,
            "mode_data": mode_data or {},
            "do_not_queue": do_not_queue,
            "robot_group_ids": robot_group_ids or [],
            "filter_category_ids": filter_category_ids or [],
            "priority": priority
        }

        return self._post(
            "/pudu-entry/open-platform-service/v1/custom_call",
            payload
        )
    

In [3]:
client = PuduApiClient(app_key=pudu_api_key, app_secret=pudu_api_secret)
print(f"PuduApiClient initialized.  {client}")

PuduApiClient initialized.  <__main__.PuduApiClient object at 0x000001CA71538980>


In [4]:
map_data = client.open_map(shop_id=408250001,map_name='1#1#內湖展間v20')
print(json.dumps(map_data, indent=2, ensure_ascii=False))

{
  "data": {
    "element_list": [
      {
        "clean_path_list": [],
        "id": "倉庫位",
        "mode": "table",
        "name": "倉庫位",
        "type": "source",
        "vector_list": [
          -12.01,
          -7.62,
          1.56
        ]
      },
      {
        "clean_path_list": [],
        "id": "喵喵充電",
        "mode": "table",
        "name": "喵喵充電",
        "type": "source",
        "vector_list": [
          -5.38,
          -7.25,
          -1.58
        ]
      },
      {
        "clean_path_list": [],
        "id": "喵喵待機",
        "mode": "dining_outlet",
        "name": "喵喵待機",
        "type": "source",
        "vector_list": [
          -2.33,
          -2.94,
          1.56
        ]
      },
      {
        "clean_path_list": [],
        "id": "堆棧輸送帶",
        "mode": "table",
        "name": "堆棧輸送帶",
        "type": "source",
        "vector_list": [
          -12.48,
          0.12,
          3.14
        ]
      },
      {
        "clean_path_list": [],

In [7]:
client = PuduApiClient(app_key=pudu_api_key, app_secret=pudu_api_secret)

# 測試 1：查詢機器人清單或狀態
# 請確認你有把 sn 替換成你們家真實的機器人序號
status = client.get_by_sn2(sn = Flashbot) 
print(status)

{'data': {'battery': 64, 'charge_stage': 'IDLE', 'charge_type': 2, 'is_charging': -1, 'mac': '98:A1:4A:77:DE:60', 'move_state': 'ARRIVE', 'product_code': 'FlashBotMax', 'remain_time': 0, 'run_state': 'BUSY', 'sn': '8FF055923050007', 'timestamp': '1783408725'}, 'message': 'SUCCESS', 'trace_id': 'APIDKWT5TRzAOgyuVr2ERhbY86J0OgS8cb8k6J7J_b34d4687-c9ec-4756-9f1f-ab50720c786b'}
